# 06 — LLM zero-shot and few-shot via OpenRouter

OpenRouter exposes an OpenAI-compatible endpoint. Set `OPENROUTER_API_KEY` in your `.env`.

**Cost control:** running zero-shot on the full test set across 4 languages can be ≥ \$10–30 depending on the model. Use `LLM_TEST_CAP` below to cap to a stratified subset for the thesis.

Model choice: cheap & strong picks at the time of writing —
  - `anthropic/claude-3.5-sonnet`
  - `openai/gpt-4o-mini`
  - `meta-llama/llama-3.1-70b-instruct`
  - `qwen/qwen-2.5-72b-instruct`

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from src import config as C
from src.data_utils import get_split
from src.evaluate import evaluate_and_log
from src.models import llm

LLM_TEST_CAP = 200            # cap test set to N stratified examples per language
MODEL_NAME   = 'anthropic/claude-3.5-sonnet'

In [ ]:
def stratified_cap(df, n):
    if n is None or n >= len(df):
        return df
    per = n // 2
    a = df[df['label'] == 0].sample(per, random_state=C.RANDOM_STATE)
    b = df[df['label'] == 1].sample(n - per, random_state=C.RANDOM_STATE)
    return pd.concat([a, b]).sample(frac=1, random_state=C.RANDOM_STATE).reset_index(drop=True)

cfg = C.LLMConfig(model=MODEL_NAME)

### Zero-shot

In [ ]:
for lang in C.LANGUAGES:
    print(f'\n=== LLM zero-shot :: {lang.upper()} ===')
    try:
        tr, va, te = get_split(lang)
    except FileNotFoundError as e:
        print(f'  skipped: {e}'); continue
    sub = stratified_cap(te, LLM_TEST_CAP)
    res = llm.classify_zero_shot(sub, lang, cfg=cfg)
    metrics = evaluate_and_log(sub['label'].values, res['pred'].values,
                                model_name='llm-zero', lang=lang,
                                extra={'llm_model': cfg.model, 'n': len(sub)})
    res.to_csv(C.RESULTS_DIR / 'tables' / f'llm_zero_{lang}_predictions.csv', index=False)
    print('  metrics:', {k: round(v, 4) for k, v in metrics.items()})

### Few-shot (k = 4 balanced)

In [ ]:
for lang in C.LANGUAGES:
    print(f'\n=== LLM few-shot :: {lang.upper()} ===')
    try:
        tr, va, te = get_split(lang)
    except FileNotFoundError as e:
        print(f'  skipped: {e}'); continue
    sub = stratified_cap(te, LLM_TEST_CAP)
    res = llm.classify_few_shot(sub, lang, train_df=tr, cfg=cfg)
    metrics = evaluate_and_log(sub['label'].values, res['pred'].values,
                                model_name='llm-few', lang=lang,
                                extra={'llm_model': cfg.model, 'k': cfg.n_few_shot, 'n': len(sub)})
    res.to_csv(C.RESULTS_DIR / 'tables' / f'llm_few_{lang}_predictions.csv', index=False)
    print('  metrics:', {k: round(v, 4) for k, v in metrics.items()})